# 📄 PaddleOCR PDF V20 — OCR GPU · pip fix

Experiencia:

1. **▶ Ejecutar**
2. elegir si querés guardar wheel + modelos en Drive;
3. aparece un **modal responsive** para subir el PDF;
4. tocar **Elegir PDF** abre el selector nativo del navegador/dispositivo;
5. muestra nombre, tamaño y progreso de subida;
6. al terminar la subida comienza el OCR automáticamente;
7. muestra página / porcentaje / tiempo / ETA;
8. descarga `nombre_OCR.pdf`.

## Subida de archivo

El modal usa un `<input type="file" accept="application/pdf">`, así que en
celular se abre el selector nativo del sistema.

La transferencia browser → kernel reutiliza el mecanismo chunked de Colab
(`google.colab._files._uploadFiles` / `_uploadFilesContinue`), el mismo
utilizado internamente por `google.colab.files.upload()`.

Los chunks se escriben directamente al disco del runtime para evitar mantener
otra copia completa del PDF en memoria Python.

**El PDF no se guarda en Google Drive.**
Drive sigue siendo opcional únicamente para los archivos pesados reutilizables
de PaddlePaddle/PaddleOCR.

## Corrección V20

Corrige el orden de argumentos de pip.

Antes:

```text
python -m pip -v --progress-bar=raw install ...
```

Ahora:

```text
python -m pip install -v --progress-bar=raw ...
```

No cambia Paddle, CUDA, PaddleOCR ni el wheel ya descargado/verificado.


In [ ]:
import os
import sys
import platform
import pathlib

print("🚀 PaddleOCR PDF V17")
print()

release = platform.release().lower()
IS_LOCAL_WSL = ("microsoft" in release) or ("wsl" in release)

HOME = pathlib.Path.home()
LOCAL_WHEEL_STORE = HOME / ".cache" / "paddle-wheel-store"
LOCAL_MODEL_CACHE = HOME / ".cache" / "paddlex"

SAVE_HEAVY_FILES = False
CACHE_MODE = "temporary"

if IS_LOCAL_WSL:
    # En nuestro Docker local /root ya es persistente.
    SAVE_HEAVY_FILES = True
    CACHE_MODE = "local-persistent"
    WHEEL_STORE = LOCAL_WHEEL_STORE
    MODEL_CACHE = LOCAL_MODEL_CACHE

    print("💾 Desarrollo local detectado.")
    print("   /root ya es persistente: no hace falta Google Drive.")
    print()

else:
    from google.colab import output

    message = """¿Querés guardar los archivos pesados de PaddleOCR en Google Drive?

Se reservarán aproximadamente 2 GB:
• PaddlePaddle GPU: ~1.76 GiB
• Modelos OCR español: ~100 MB aprox.

RECOMENDADO: Aceptar
Evita volver a descargar ~2 GB en futuras sesiones.

Cancelar = usar solo esta sesión.
El OCR funciona igual, pero en otra sesión probablemente haya que descargar todo de nuevo.

Si tu cuenta institucional de la facultad tiene más almacenamiento, recomendamos usar esa cuenta.

Tus PDFs NO se guardan en Drive.
Solo se guardan el motor y los modelos reutilizables."""

    # Native browser confirmation.
    # True  => OK / Aceptar => persist to Drive
    # False => Cancel / Cancelar => temporary session
    SAVE_HEAVY_FILES = bool(
        output.eval_js(
            "window.confirm(" + repr(message) + ")"
        )
    )

    if SAVE_HEAVY_FILES:
        from google.colab import drive

        print()
        print("✅ Elegiste guardar los archivos pesados.")
        print("🔗 Montando Google Drive...")
        drive.mount("/content/drive", force_remount=False)

        DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/PaddleOCR")
        WHEEL_STORE = DRIVE_ROOT / "wheels"
        MODEL_CACHE = DRIVE_ROOT / "modelos"
        CACHE_MODE = "google-drive"

        print()
        print("✅ Caché persistente:")
        print("   Mi unidad/PaddleOCR/")
    else:
        WHEEL_STORE = LOCAL_WHEEL_STORE
        MODEL_CACHE = LOCAL_MODEL_CACHE
        CACHE_MODE = "temporary"

        print()
        print("✅ Elegiste usar solo esta sesión.")
        print("   El OCR funciona normalmente.")
        print("   Nada pesado se guardará en Drive.")
        print("   En otra sesión probablemente haya que descargar")
        print("   nuevamente ~1.9–2.0 GB.")

for d in (WHEEL_STORE, MODEL_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ["PADDLEOCR_CACHE_MODE"] = CACHE_MODE
os.environ["PADDLEOCR_WHEEL_STORE"] = str(WHEEL_STORE)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "bos"

print()
print("Modo de caché:", CACHE_MODE)
print("Wheel:", WHEEL_STORE)
print("Modelos:", MODEL_CACHE)
print()

In [ ]:
import sys
import os
import pathlib
import subprocess
import importlib.metadata as md
import platform
import shutil
import time
import re
import zlib
import json
import threading
import queue
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

PADDLE = "3.2.0"
OCR = "3.2.0"

PADDLE_WHEEL_URL = (
    "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
)

WHEEL_NAME = "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"

CHUNK_SIZE = 64 * 1024 * 1024
PARALLEL_DOWNLOADS = 4
CHUNK_MAX_TIME = 180
LOW_SPEED_LIMIT = 32 * 1024
LOW_SPEED_TIME = 20

if platform.system().lower() != "linux":
    raise RuntimeError("Este notebook espera un runtime Linux de Google Colab.")

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Este build usa Python 3.12; encontré "
        f"{sys.version_info.major}.{sys.version_info.minor}."
    )

HOME = pathlib.Path.home()

WHEEL_STORE = pathlib.Path(
    os.environ.get(
        "PADDLEOCR_WHEEL_STORE",
        str(HOME / ".cache" / "paddle-wheel-store"),
    )
)
MODEL_CACHE = pathlib.Path(
    os.environ.get(
        "PADDLE_PDX_CACHE_HOME",
        str(HOME / ".cache" / "paddlex"),
    )
)
CACHE_MODE = os.environ.get("PADDLEOCR_CACHE_MODE", "temporary")

PIP_CACHE = HOME / ".cache" / "pip"

# Chunk cache stays local to the runtime. Only the FINAL verified wheel
# is copied to Drive when Drive persistence is selected.
CHUNK_ROOT = HOME / ".cache" / "paddle-wheel-chunks" / "3.2.0-cu126-cp312"

for d in (WHEEL_STORE, MODEL_CACHE, PIP_CACHE, CHUNK_ROOT):
    d.mkdir(parents=True, exist_ok=True)

os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "bos"

PERSIST_WHEEL = WHEEL_STORE / WHEEL_NAME
PERSIST_MANIFEST = WHEEL_STORE / f"{WHEEL_NAME}.verified.json"
TMP_WHEEL = pathlib.Path("/tmp") / WHEEL_NAME


def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


def human_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units) - 1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"


def run_pip_live(args, title):
    args = list(map(str, args))

    # pip's --progress-bar is an INSTALL subcommand option, not a global option.
    # Correct:
    #   python -m pip install -v --progress-bar=raw ...
    # Wrong:
    #   python -m pip -v --progress-bar=raw install ...
    if args and args[0] == "install":
        cmd = [
            sys.executable, "-m", "pip",
            "install",
            "-v",
            "--progress-bar=raw",
            *args[1:],
        ]
    else:
        cmd = [
            sys.executable, "-m", "pip",
            *args,
        ]

    print()
    print("=" * 88)
    print(f"📦 {title}")
    print("=" * 88)
    print("$", " ".join(cmd), flush=True)
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    q = queue.Queue()

    def reader():
        try:
            for line in iter(proc.stdout.readline, ""):
                q.put(line)
        finally:
            q.put(None)

    threading.Thread(target=reader, daemon=True).start()

    progress_re = re.compile(r"Progress\s+(\d+)\s+of\s+(\d+)", re.I)

    started = time.time()
    last_output = started
    reader_done = False
    current = 0
    total = None
    last_progress_print = 0.0

    while proc.poll() is None or not reader_done:
        while True:
            try:
                item = q.get_nowait()
            except queue.Empty:
                break

            if item is None:
                reader_done = True
                break

            last_output = time.time()
            line = item.rstrip("\r\n")

            m = progress_re.search(line)
            if m:
                current = int(m.group(1))
                total = int(m.group(2))
                now = time.time()

                if now - last_progress_print >= 0.5:
                    pct = current / total * 100 if total else 0
                    print(
                        f"[pip descarga] {human_bytes(current)} / "
                        f"{human_bytes(total)} ({pct:.1f}%)",
                        flush=True,
                    )
                    last_progress_print = now
            elif line.strip():
                print(f"[pip] {line}", flush=True)

        now = time.time()

        if proc.poll() is None and now - last_output >= 3:
            elapsed = now - started

            if total:
                pct = current / total * 100 if total else 0
                print(
                    f"[{elapsed:6.1f}s] ⏳ pip sigue trabajando · "
                    f"{human_bytes(current)} / {human_bytes(total)} "
                    f"({pct:.1f}%)",
                    flush=True,
                )
            else:
                print(
                    f"[{elapsed:6.1f}s] ⏳ pip sigue trabajando "
                    f"(resolviendo/descomprimiendo/instalando)...",
                    flush=True,
                )
            last_output = now

        time.sleep(0.15)

    rc = proc.wait()
    elapsed = time.time() - started

    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

    print()
    print(f"✅ {title} terminado en {elapsed:.1f}s.")


def remote_metadata(url):
    curl = shutil.which("curl")

    p = subprocess.run(
        [
            curl,
            "--location",
            "--silent",
            "--show-error",
            "--fail",
            "--range", "0-0",
            "--dump-header", "-",
            "--output", "/dev/null",
            url,
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    headers = p.stdout

    totals = re.findall(
        r"(?im)^content-range:\s*bytes\s+\d+-\d+/(\d+)\s*$",
        headers,
    )
    crcs = re.findall(
        r"(?im)^x-bce-content-crc32:\s*(\d+)\s*$",
        headers,
    )
    etags = re.findall(
        r'(?im)^etag:\s*"?([^"\r\n]+)"?\s*$',
        headers,
    )

    if not totals or not crcs:
        raise RuntimeError("El CDN no informó tamaño/CRC32 del wheel.")

    return {
        "total": int(totals[-1]),
        "crc32": int(crcs[-1]),
        "etag": etags[-1] if etags else None,
    }


def build_chunks(total):
    chunks = []
    start = 0
    idx = 0

    while start < total:
        end = min(total - 1, start + CHUNK_SIZE - 1)
        chunks.append((idx, start, end))
        idx += 1
        start = end + 1

    return chunks


def chunk_path(idx, start, end):
    return CHUNK_ROOT / f"{idx:04d}_{start:012d}-{end:012d}.bin"


def chunk_is_complete(idx, start, end):
    p = chunk_path(idx, start, end)
    return p.exists() and p.stat().st_size == (end - start + 1)


def download_one_chunk(item):
    idx, start, end = item
    final = chunk_path(idx, start, end)
    expected = end - start + 1

    if chunk_is_complete(idx, start, end):
        return idx, expected, "cached"

    tmp = final.with_suffix(".tmp")
    headers_path = final.with_suffix(".headers")

    for p in (tmp, headers_path):
        if p.exists():
            p.unlink()

    cmd = [
        shutil.which("curl"),
        "--location",
        "--fail",
        "--silent",
        "--show-error",
        "--retry", "10",
        "--retry-delay", "1",
        "--retry-all-errors",
        "--connect-timeout", "30",
        "--max-time", str(CHUNK_MAX_TIME),
        "--speed-limit", str(LOW_SPEED_LIMIT),
        "--speed-time", str(LOW_SPEED_TIME),
        "--range", f"{start}-{end}",
        "--dump-header", str(headers_path),
        "--output", str(tmp),
        PADDLE_WHEEL_URL,
    ]

    p = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )

    hdr = headers_path.read_text(errors="replace") if headers_path.exists() else ""
    got = tmp.stat().st_size if tmp.exists() else 0

    statuses = re.findall(r"HTTP/\S+\s+(\d+)", hdr)
    ranges = re.findall(
        r"(?im)^content-range:\s*bytes\s+(\d+)-(\d+)/(\d+)\s*$",
        hdr,
    )

    range_ok = bool(
        ranges
        and int(ranges[-1][0]) == start
        and int(ranges[-1][1]) == end
    )

    if p.returncode != 0 or "206" not in statuses or not range_ok or got != expected:
        if tmp.exists():
            tmp.unlink()

        raise RuntimeError(
            f"bloque {idx+1}: rc={p.returncode}, "
            f"HTTP={statuses}, range={ranges[-1:]}, "
            f"bytes={got}/{expected}, stderr={p.stderr[-300:]}"
        )

    tmp.rename(final)

    if headers_path.exists():
        headers_path.unlink()

    return idx, expected, "downloaded"


def download_all_chunks(chunks):
    total = sum(end - start + 1 for _, start, end in chunks)
    cached = sum(
        end - start + 1
        for idx, start, end in chunks
        if chunk_is_complete(idx, start, end)
    )

    missing = [
        item
        for item in chunks
        if not chunk_is_complete(*item)
    ]

    print()
    print("=== DESCARGA SEGURA DEL WHEEL ===")
    print("Bloques totales:      ", len(chunks))
    print("Bloques ya completos: ", len(chunks) - len(missing))
    print("Bloques por descargar:", len(missing))
    print("Cacheado:              ", human_bytes(cached))
    print("Total:                 ", human_bytes(total))
    print()

    if not missing:
        print("✅ Todos los bloques ya están disponibles.")
        return

    done_bytes = cached
    started = time.time()
    errors = []

    bar = tqdm(
        total=total,
        initial=cached,
        desc="Paddle wheel",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
    )

    with ThreadPoolExecutor(max_workers=PARALLEL_DOWNLOADS) as pool:
        future_map = {
            pool.submit(download_one_chunk, item): item
            for item in missing
        }

        pending = set(future_map)
        last_status = time.time()

        while pending:
            finished = {f for f in pending if f.done()}

            if not finished:
                now = time.time()

                if now - last_status >= 5:
                    ids = sorted(future_map[f][0] + 1 for f in pending)
                    print(
                        f"[{now-started:6.1f}s] ⏳ esperando bloques {ids}",
                        flush=True,
                    )
                    last_status = now

                time.sleep(0.5)
                continue

            for fut in finished:
                pending.remove(fut)
                idx, start, end = future_map[fut]

                try:
                    _, amount, _ = fut.result()
                    done_bytes += amount
                    bar.update(amount)

                    elapsed = max(0.001, time.time() - started)
                    speed = max(0, done_bytes - cached) / elapsed
                    remaining = max(0, total - done_bytes)
                    eta = remaining / speed if speed > 0 else None

                    eta_text = (
                        f"{eta/60:.1f} min" if eta is not None and eta >= 60
                        else f"{eta:.0f} s" if eta is not None
                        else "calculando"
                    )

                    bar.set_postfix_str(
                        f"bloque {idx+1}/{len(chunks)} · "
                        f"{human_bytes(speed)}/s · ETA {eta_text}"
                    )

                    print(
                        f"✅ bloque {idx+1:02d}/{len(chunks)} · "
                        f"{done_bytes/total*100:5.1f}%",
                        flush=True,
                    )

                except Exception as e:
                    errors.append((idx, str(e)))
                    print(f"❌ bloque {idx+1}: {e}", flush=True)

    bar.close()

    if errors:
        print()
        print("Los bloques correctos quedan cacheados en este runtime.")
        raise RuntimeError(
            f"Fallaron {len(errors)} bloque(s). "
            "Volvé a ejecutar esta celda para reintentar solo los faltantes."
        )


def assemble_and_crc(chunks, expected_total, expected_crc):
    if TMP_WHEEL.exists():
        TMP_WHEEL.unlink()

    crc = 0
    written = 0

    print()
    print("=== ENSAMBLANDO + VERIFICANDO CRC32 ===")

    with TMP_WHEEL.open("wb") as out, tqdm(
        total=expected_total,
        desc="Ensamblando",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
    ) as bar:

        for idx, start, end in chunks:
            p = chunk_path(idx, start, end)

            with p.open("rb") as inp:
                while True:
                    data = inp.read(16 * 1024 * 1024)
                    if not data:
                        break

                    out.write(data)
                    crc = zlib.crc32(data, crc)
                    written += len(data)
                    bar.update(len(data))

    crc &= 0xFFFFFFFF

    print("Tamaño:", written, "/", expected_total)
    print("CRC32 local:  ", crc)
    print("CRC32 oficial:", expected_crc)

    if written != expected_total or crc != expected_crc:
        raise RuntimeError("El wheel ensamblado no pasó la verificación.")

    print("✅ Wheel íntegro.")


def validate_zip(path):
    unzip = shutil.which("unzip")

    print()
    print("🔎 Validando ZIP/wheel...")

    if unzip:
        p = subprocess.run(
            [unzip, "-tqq", str(path)],
            capture_output=True,
            text=True,
        )

        if p.returncode != 0:
            raise RuntimeError(
                "El wheel no pasó unzip -t:\n"
                + (p.stdout + "\n" + p.stderr)[-1500:]
            )

        print("✅ ZIP válido.")
        return

    p = subprocess.run(
        [sys.executable, "-m", "zipfile", "-t", str(path)],
        capture_output=True,
        text=True,
    )

    if p.returncode != 0:
        raise RuntimeError("El wheel no pasó zipfile -t.")

    print("✅ ZIP válido.")


def manifest_matches(meta):
    if not PERSIST_WHEEL.exists() or not PERSIST_MANIFEST.exists():
        return False

    try:
        saved = json.loads(PERSIST_MANIFEST.read_text(encoding="utf-8"))
    except Exception:
        return False

    return (
        PERSIST_WHEEL.stat().st_size == meta["total"]
        and saved.get("total") == meta["total"]
        and saved.get("crc32") == meta["crc32"]
        and saved.get("etag") == meta["etag"]
    )


def copy_with_progress(src, dst, title):
    src = pathlib.Path(src)
    dst = pathlib.Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    tmp = dst.with_suffix(dst.suffix + ".copying")
    if tmp.exists():
        tmp.unlink()

    total = src.stat().st_size

    print()
    print(title)

    with src.open("rb") as inp, tmp.open("wb") as out, tqdm(
        total=total,
        desc="Copiando",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
    ) as bar:
        while True:
            data = inp.read(16 * 1024 * 1024)
            if not data:
                break
            out.write(data)
            bar.update(len(data))

    if tmp.stat().st_size != total:
        raise RuntimeError("La copia terminó con tamaño incorrecto.")

    tmp.replace(dst)


print("🧩 Preparando PaddleOCR")
print("Modo de caché:", CACHE_MODE)
print("Wheel persistente:", PERSIST_WHEEL)
print("Modelos:", MODEL_CACHE)
print()

# ---------------------------------------------------------------
# PaddlePaddle GPU
# ---------------------------------------------------------------
if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya está instalado.")

else:
    existing = ver("paddlepaddle-gpu")

    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing}; esperaba {PADDLE}."
        )

    print("🌐 Verificando metadata oficial del wheel...")
    meta = remote_metadata(PADDLE_WHEEL_URL)

    print("Tamaño:", human_bytes(meta["total"]))
    print("CRC32:", meta["crc32"])
    print("ETag:", meta["etag"])

    if manifest_matches(meta):
        print()
        print("✅ Wheel verificado encontrado en la caché persistente.")
        print("   No se vuelve a descargar desde Internet.")

        copy_with_progress(
            PERSIST_WHEEL,
            TMP_WHEEL,
            "📥 Copiando wheel cacheado al runtime...",
        )

        validate_zip(TMP_WHEEL)

    else:
        if PERSIST_WHEEL.exists():
            print()
            print("⚠️ Hay un wheel guardado, pero no coincide con el manifiesto")
            print("   esperado. No lo voy a usar.")

        chunks = build_chunks(meta["total"])
        download_all_chunks(chunks)
        assemble_and_crc(
            chunks,
            expected_total=meta["total"],
            expected_crc=meta["crc32"],
        )
        validate_zip(TMP_WHEEL)

        # Save the heavy wheel only when persistence was requested.
        # Local development is also persistent by design.
        if CACHE_MODE in {"google-drive", "local-persistent"}:
            copy_with_progress(
                TMP_WHEEL,
                PERSIST_WHEEL,
                "💾 Guardando wheel verificado para futuros usos...",
            )

            PERSIST_MANIFEST.write_text(
                json.dumps(meta, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            print("✅ Wheel pesado guardado en la caché persistente.")
        else:
            print()
            print("ℹ️ Elegiste NO guardar los archivos pesados.")
            print("   El wheel sirve para esta sesión, pero no se copiará a Drive.")

    run_pip_live(
        [
            "install",
            "--retries", "10",
            "--timeout", "180",
            str(TMP_WHEEL),
        ],
        "PaddlePaddle GPU 3.2.0",
    )

# ---------------------------------------------------------------
# PaddleOCR package
# ---------------------------------------------------------------
if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya está instalado.")
else:
    existing = ver("paddleocr")

    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing}; esperaba {OCR}."
        )

    run_pip_live(
        [
            "install",
            "--retries", "10",
            "--timeout", "180",
            f"paddleocr=={OCR}",
        ],
        "PaddleOCR 3.2.0",
    )

print()
print("✅ Motor listo.")
if CACHE_MODE == "google-drive":
    print("💾 Wheel + modelos configurados para persistir en Google Drive.")
elif CACHE_MODE == "temporary":
    print("ℹ️ Modo temporal: funciona ahora, pero futuras sesiones")
    print("   probablemente deberán descargar nuevamente los archivos pesados.")
else:
    print("💾 Caché local persistente activa.")

In [ ]:
# ================================================================
# ▶ SUBIR PDF → OCR GPU → DESCARGAR PDF SEARCHABLE
# ================================================================
#
# Podés volver a ejecutar SOLO esta celda para procesar otro PDF.
#
# Ajustes principales:
DPI = 250
MIN_SCORE = 0.30
SKIP_ALREADY_SEARCHABLE = True
EXISTING_TEXT_MIN_CHARS = 80

import os
import sys
import re
import time
import pathlib
import subprocess
import importlib.metadata as md
import platform

print("📄 OCR DE PDF")
print()

# ================================================================
# 💾 CACHE YA ELEGIDA EN LA PRIMERA CELDA
# ================================================================
CACHE_MODE = os.environ.get("PADDLEOCR_CACHE_MODE", "temporary")
MODEL_CACHE = pathlib.Path(
    os.environ.get(
        "PADDLE_PDX_CACHE_HOME",
        str(pathlib.Path.home() / ".cache" / "paddlex"),
    )
)

os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "bos"

print("💾 Modo de caché:", CACHE_MODE)
print("   Modelos:", MODEL_CACHE)
print()

# ------------------------------------------------
# Ensure PyMuPDF. It is loaded only in the worker.
# ------------------------------------------------
try:
    pymupdf_version = md.version("PyMuPDF")
    print("✅ PyMuPDF:", pymupdf_version)
except md.PackageNotFoundError:
    print("📦 PyMuPDF no está instalado. Instalando 1.26.4...")
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "pip", "install",
            "-v",
            "--progress-bar=raw",
            "PyMuPDF==1.26.4",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print("[pip]", line, end="", flush=True)
    if proc.wait():
        raise RuntimeError("Falló la instalación de PyMuPDF.")
    print("✅ PyMuPDF instalado.")

# ------------------------------------------------
# Responsive browser modal using Colab's own chunked uploader.
# ------------------------------------------------
try:
    from google.colab import files, output
    import google.colab.files as colab_files_module
except Exception as e:
    raise RuntimeError(
        "No encuentro las APIs de Google Colab. "
        "Abrí este notebook desde la interfaz de Google Colab, "
        "aunque el runtime sea local."
    ) from e

import base64
import pkgutil
import uuid
import json
from IPython.display import display, HTML

upload_dir = pathlib.Path.home() / ".cache" / "paddleocr-upload"
output_dir = pathlib.Path.home() / ".cache" / "paddleocr-output"
dev_dir = pathlib.Path.home() / ".cache" / "paddleocr-dev"

for d in (upload_dir, output_dir, dev_dir):
    d.mkdir(parents=True, exist_ok=True)


def upload_pdf_modal(target_dir):
    """
    Modal visual + selector nativo del navegador.

    El transporte reutiliza google.colab._files._uploadFiles(),
    el mismo uploader chunked que usa google.colab.files.upload().

    Los chunks se escriben directamente al disco para no duplicar
    el PDF entero en memoria Python.
    """
    upload_id = str(uuid.uuid4()).replace("-", "")
    input_id = f"paddle-pdf-input-{upload_id}"
    output_id = f"paddle-pdf-output-{upload_id}"
    dialog_id = f"paddle-pdf-dialog-{upload_id}"
    status_id = f"paddle-pdf-status-{upload_id}"
    label_id = f"paddle-pdf-label-{upload_id}"

    files_js = pkgutil.get_data(
        colab_files_module.__name__,
        "resources/files.js",
    ).decode("utf-8")

    modal_html = f"""
    <style>
      #{dialog_id} {{
        width: min(92vw, 680px);
        max-width: 680px;
        border: 0;
        border-radius: 22px;
        padding: 0;
        background: transparent;
        color: inherit;
      }}
      #{dialog_id}::backdrop {{
        background: rgba(0,0,0,.52);
        backdrop-filter: blur(2px);
      }}
      #{dialog_id} .ocr-card {{
        box-sizing: border-box;
        width: 100%;
        padding: clamp(18px, 5vw, 30px);
        border-radius: 22px;
        background: var(--colab-primary-surface-color, #fff);
        color: var(--colab-primary-text-color, #202124);
        box-shadow: 0 16px 50px rgba(0,0,0,.24);
        font-family: Arial, sans-serif;
      }}
      #{dialog_id} .ocr-title {{
        font-size: clamp(21px, 5vw, 28px);
        line-height: 1.2;
        font-weight: 750;
        margin: 0 0 8px 0;
      }}
      #{dialog_id} .ocr-subtitle {{
        font-size: clamp(14px, 3.7vw, 17px);
        line-height: 1.45;
        opacity: .78;
        margin-bottom: 18px;
      }}
      #{dialog_id} .picker {{
        display: flex;
        align-items: center;
        justify-content: center;
        width: 100%;
        min-height: 64px;
        box-sizing: border-box;
        padding: 14px 18px;
        border-radius: 15px;
        border: 2px solid #1a73e8;
        background: #1a73e8;
        color: white;
        font-size: clamp(16px, 4vw, 19px);
        line-height: 1.2;
        font-weight: 700;
        cursor: pointer;
        text-align: center;
        touch-action: manipulation;
        user-select: none;
        -webkit-tap-highlight-color: transparent;
      }}
      #{dialog_id} .picker:active {{
        transform: scale(.99);
      }}
      #{dialog_id} input[type=file] {{
        position: absolute !important;
        width: 1px !important;
        height: 1px !important;
        opacity: 0 !important;
        pointer-events: none !important;
      }}
      #{dialog_id} .file-status {{
        box-sizing: border-box;
        width: 100%;
        margin-top: 14px;
        padding: 13px 15px;
        border-radius: 13px;
        background: rgba(26,115,232,.08);
        font-size: clamp(13px, 3.5vw, 15px);
        line-height: 1.45;
        word-break: break-word;
      }}
      #{dialog_id} .privacy {{
        margin-top: 14px;
        font-size: clamp(12px, 3.2vw, 14px);
        line-height: 1.45;
        opacity: .72;
      }}
      #{dialog_id} #{output_id} {{
        display: block;
        margin: 12px 0 0 0;
        padding: 0;
        font-size: 13px;
        line-height: 1.4;
      }}
      #{dialog_id} #{output_id} li {{
        list-style: none;
        box-sizing: border-box;
        padding: 10px 12px;
        margin: 8px 0 0 0;
        border-radius: 10px;
        background: rgba(52,168,83,.10);
        word-break: break-word;
      }}
      #{dialog_id} .upload-host > button {{
        width: 100%;
        min-height: 44px;
        margin-top: 10px;
        border: 1px solid rgba(128,128,128,.4);
        border-radius: 11px;
        background: transparent;
        color: inherit;
        font-size: 14px;
        cursor: pointer;
      }}
      @media (max-width: 480px) {{
        #{dialog_id} {{
          width: 94vw;
        }}
        #{dialog_id} .ocr-card {{
          padding: 18px;
          border-radius: 18px;
        }}
      }}
    </style>

    <dialog id="{dialog_id}">
      <div class="ocr-card">
        <div class="ocr-title">📄 Subí el PDF que querés hacer searchable</div>
        <div class="ocr-subtitle">
          Seleccioná un único PDF. Cuando termine de subir,
          el OCR empieza automáticamente.
        </div>

        <div class="upload-host">
          <label class="picker" for="{input_id}" id="{label_id}">
            Elegir PDF
          </label>
          <input
            type="file"
            id="{input_id}"
            name="files[]"
            accept="application/pdf,.pdf"
            disabled
          />
        </div>

        <div class="file-status" id="{status_id}">
          Ningún archivo seleccionado.
        </div>

        <output id="{output_id}"></output>

        <div class="privacy">
          🔒 El PDF se usa en el runtime temporal para hacer OCR.
          No se guarda en Google Drive.
        </div>
      </div>
    </dialog>

    <script>
    {files_js}

    (() => {{
      const dialog = document.getElementById({json.dumps(dialog_id)});
      const input = document.getElementById({json.dumps(input_id)});
      const status = document.getElementById({json.dumps(status_id)});
      const label = document.getElementById({json.dumps(label_id)});

      const fmt = (bytes) => {{
        const units = ['B','KB','MB','GB'];
        let n = bytes;
        let i = 0;
        while (n >= 1024 && i < units.length - 1) {{
          n /= 1024;
          i++;
        }}
        return `${{n.toFixed(i === 0 ? 0 : 1)}} ${{units[i]}}`;
      }};

      input.addEventListener('change', () => {{
        const file = input.files && input.files[0];
        if (!file) {{
          status.textContent = 'Ningún archivo seleccionado.';
          return;
        }}

        const isPdf =
          file.type === 'application/pdf' ||
          file.name.toLowerCase().endsWith('.pdf');

        if (!isPdf) {{
          status.innerHTML = '❌ <b>Ese archivo no parece ser un PDF.</b>';
          input.value = '';
          return;
        }}

        const safeName = file.name.replace(/[&<>"']/g, '');
        status.innerHTML =
          '✅ <b>' + safeName + '</b><br>' +
          fmt(file.size) + ' · subiendo al runtime…';
        label.textContent = 'PDF seleccionado';
      }});

      if (typeof dialog.showModal === 'function') {{
        dialog.showModal();
      }} else {{
        dialog.setAttribute('open', '');
      }}
    }})();
    </script>
    """

    display(HTML(modal_html))

    result = output.eval_js(
        "google.colab._files._uploadFiles("
        + json.dumps(input_id)
        + ", "
        + json.dumps(output_id)
        + ")"
    )

    saved_path = None
    saved_name = None
    fh = None

    try:
        while result["action"] != "complete":
            result = output.eval_js(
                "google.colab._files._uploadFilesContinue("
                + json.dumps(output_id)
                + ")"
            )

            if result["action"] != "append":
                continue

            incoming_name = pathlib.Path(str(result["file"])).name

            if not incoming_name.lower().endswith(".pdf"):
                raise RuntimeError("El archivo seleccionado no es un PDF.")

            if saved_path is None:
                saved_name = incoming_name
                saved_path = pathlib.Path(target_dir) / saved_name

                if saved_path.exists():
                    saved_path.unlink()

                fh = saved_path.open("ab")

            elif incoming_name != saved_name:
                raise RuntimeError("Esta versión acepta un solo PDF por vez.")

            fh.write(base64.b64decode(result["data"]))

    finally:
        if fh is not None:
            fh.close()

        try:
            close_js = (
                "(() => {"
                "const d=document.getElementById("
                + json.dumps(dialog_id)
                + ");"
                "if(d && d.open){d.close();}"
                "if(d){d.remove();}"
                "return true;"
                "})()"
            )
            output.eval_js(close_js)
        except Exception:
            pass

    if saved_path is None or not saved_path.exists():
        raise RuntimeError("No se seleccionó ningún PDF.")

    with saved_path.open("rb") as f:
        signature = f.read(5)

    if signature != b"%PDF-":
        saved_path.unlink(missing_ok=True)
        raise RuntimeError(
            "El archivo tiene extensión .pdf pero no tiene firma PDF válida."
        )

    return saved_path


print()
input_path = upload_pdf_modal(upload_dir)
safe_name = input_path.name

print()
print("✅ PDF recibido:", safe_name)
print(f"   {input_path.stat().st_size / (1024*1024):.1f} MB")
print()

stem = pathlib.Path(safe_name).stem
output_path = output_dir / f"{stem}_OCR.pdf"

if output_path.exists():
    output_path.unlink()

print()
print("✅ PDF recibido:", safe_name)
print(f"   {input_path.stat().st_size / (1024*1024):.1f} MB")
print()

# ------------------------------------------------
# Fresh worker: avoids binary-package kernel issues.
# ------------------------------------------------
worker_path = dev_dir / "paddle_pdf_worker_v13.py"
worker_path.write_text('import os\nimport sys\nimport json\nimport time\nimport types\nimport argparse\nimport statistics\nimport importlib.metadata as importlib_metadata\nfrom pathlib import Path\n\nos.environ["PADDLE_PDX_MODEL_SOURCE"] = "bos"\nos.environ.setdefault(\n    "PADDLE_PDX_CACHE_HOME",\n    str(Path.home() / ".cache" / "paddlex"),\n)\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--input", required=True)\nparser.add_argument("--output", required=True)\nparser.add_argument("--dpi", type=int, default=250)\nparser.add_argument("--min-score", type=float, default=0.30)\nparser.add_argument("--skip-searchable", type=int, default=1)\nparser.add_argument("--existing-text-min", type=int, default=80)\nargs = parser.parse_args()\n\nINPUT = Path(args.input)\nOUTPUT = Path(args.output)\nDPI = args.dpi\nMIN_SCORE = args.min_score\nSKIP_SEARCHABLE = bool(args.skip_searchable)\nEXISTING_TEXT_MIN = args.existing_text_min\n\nprint("=== PaddleOCR PDF V13 ===", flush=True)\nprint("Entrada:", INPUT, flush=True)\nprint("Salida: ", OUTPUT, flush=True)\nprint("DPI:", DPI, flush=True)\nprint("Score mínimo:", MIN_SCORE, flush=True)\nprint("Saltar páginas ya searchables:", SKIP_SEARCHABLE, flush=True)\nprint("Cache modelos:", os.environ["PADDLE_PDX_CACHE_HOME"], flush=True)\nprint()\n\n# ---------------------------------------------------------------------\n# Compatibility guards already validated in V12.\n# ---------------------------------------------------------------------\nmodelscope_stub = types.ModuleType("modelscope")\n\ndef _modelscope_disabled(*args, **kwargs):\n    raise RuntimeError(\n        "ModelScope está deshabilitado para este runtime OCR. "\n        "Los modelos están fijados a BOS."\n    )\n\nmodelscope_stub.snapshot_download = _modelscope_disabled\nmodelscope_stub.__version__ = "disabled-by-ocr-runtime"\nsys.modules["modelscope"] = modelscope_stub\n\n_original_version = importlib_metadata.version\n\ndef _version_for_ocr_import(distribution_name):\n    normalized = str(distribution_name).strip().lower().replace("_", "-")\n    if normalized in {"langchain", "langchain-community"}:\n        raise importlib_metadata.PackageNotFoundError(distribution_name)\n    return _original_version(distribution_name)\n\nimportlib_metadata.version = _version_for_ocr_import\n\nimport paddle\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve una GPU CUDA.")\n\npaddle.set_device("gpu:0")\n\ntry:\n    gpu_name = paddle.device.cuda.get_device_name()\nexcept Exception:\n    gpu_name = "gpu:0"\n\nprint("Paddle:", paddle.__version__, flush=True)\nprint("GPU:", gpu_name, flush=True)\n\nimport numpy as np\nimport pymupdf\n\nprint("PyMuPDF:", pymupdf.__version__, flush=True)\nprint("Importando PaddleOCR...", flush=True)\n\ntry:\n    from paddleocr import PaddleOCR\n    import paddleocr\nfinally:\n    importlib_metadata.version = _original_version\n\nif "torch" in sys.modules:\n    raise RuntimeError(\n        "Torch fue importado inesperadamente. "\n        "Abortando para no reintroducir el conflicto NCCL."\n    )\n\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\nprint("✅ Stack OCR cargado sin Torch.", flush=True)\nprint()\n\n# ---------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------\ndef fmt_seconds(seconds):\n    if seconds is None:\n        return "calculando"\n    seconds = max(0, int(round(seconds)))\n    if seconds < 60:\n        return f"{seconds}s"\n    m, s = divmod(seconds, 60)\n    if m < 60:\n        return f"{m}m {s:02d}s"\n    h, m = divmod(m, 60)\n    return f"{h}h {m:02d}m"\n\ndef fmt_bytes(n):\n    n = float(n)\n    units = ["B", "KB", "MB", "GB"]\n    i = 0\n    while n >= 1024 and i < len(units) - 1:\n        n /= 1024\n        i += 1\n    return f"{n:.1f} {units[i]}"\n\ndef ascii_bar(done, total, width=24):\n    frac = done / total if total else 1.0\n    filled = max(0, min(width, int(round(frac * width))))\n    return "█" * filled + "░" * (width - filled)\n\ndef result_dict(result):\n    d = getattr(result, "json", result)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    return d if isinstance(d, dict) else {}\n\ndef clean_text(text):\n    # Avoid control characters in PDF content streams.\n    text = str(text).replace("\\x00", "")\n    text = " ".join(text.split())\n    return text.strip()\n\ndef map_pix_box_to_page(page, pix, box):\n    """\n    Paddle boxes are in rendered-pixmap coordinates.\n    page.rect -> pix.irect maps rotated page view to pixmap.\n    Invert that, then derotate because insertion coordinates must\n    be in the unrotated page coordinate system.\n    """\n    x0, y0, x1, y1 = [float(v) for v in box]\n    r_pix = pymupdf.Rect(x0, y0, x1, y1)\n\n    page_to_pix = page.rect.torect(pix.irect)\n    pix_to_rotated_page = ~page_to_pix\n    r = r_pix * pix_to_rotated_page\n\n    if page.rotation:\n        r = r * page.derotation_matrix\n\n    r = pymupdf.Rect(r)\n    r.normalize()\n\n    # Keep a tiny valid rectangle; pathological OCR boxes are ignored.\n    if r.width <= 0.5 or r.height <= 0.5:\n        return None\n\n    return r\n\ndef install_page_font(page):\n    """\n    Unicode-capable font for OCR layer.\n    DejaVu is present in the official Colab runtime.\n    Falls back to Helvetica if unavailable.\n    """\n    candidates = [\n        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",\n    ]\n\n    for candidate in candidates:\n        if Path(candidate).exists():\n            try:\n                page.insert_font(\n                    fontname="OCRFONT",\n                    fontfile=candidate,\n                    set_simple=False,\n                )\n                return "OCRFONT"\n            except Exception:\n                pass\n\n    # Base14 fallback: good for normal Spanish / Latin text.\n    return "helv"\n\ndef insert_hidden_text(page, rect, text, fontname):\n    """\n    Add PDF text with rendering mode 3 = invisible.\n    Reduce fontsize until the line fits its OCR box.\n    """\n    text = clean_text(text)\n    if not text or rect is None:\n        return False\n\n    # Slight vertical breathing room improves textbox fitting.\n    h = max(1.0, rect.height)\n    expanded = pymupdf.Rect(\n        rect.x0,\n        max(0, rect.y0 - 0.08 * h),\n        rect.x1,\n        rect.y1 + 0.18 * h,\n    )\n\n    fontsize = max(2.5, min(48.0, h * 0.78))\n\n    for _ in range(8):\n        try:\n            rc = page.insert_textbox(\n                expanded,\n                text,\n                fontname=fontname,\n                fontsize=fontsize,\n                align=pymupdf.TEXT_ALIGN_LEFT,\n                render_mode=3,\n                overlay=True,\n            )\n        except Exception:\n            rc = -1\n\n        if rc >= 0:\n            return True\n\n        fontsize *= 0.82\n\n    # Last-resort invisible baseline insertion.\n    try:\n        baseline = pymupdf.Point(\n            rect.x0,\n            max(rect.y0 + 2.0, rect.y1 - 0.12 * h),\n        )\n        page.insert_text(\n            baseline,\n            text,\n            fontname=fontname,\n            fontsize=max(2.0, fontsize),\n            render_mode=3,\n            overlay=True,\n        )\n        return True\n    except Exception:\n        return False\n\n# ---------------------------------------------------------------------\n# Open PDF and initialize OCR once.\n# ---------------------------------------------------------------------\nif not INPUT.exists():\n    raise FileNotFoundError(INPUT)\n\ndoc = pymupdf.open(INPUT)\n\nif doc.needs_pass:\n    raise RuntimeError(\n        "El PDF está protegido con contraseña. "\n        "Esta versión todavía no solicita passwords."\n    )\n\npages = len(doc)\nif pages == 0:\n    raise RuntimeError("El PDF no tiene páginas.")\n\nprint(f"📄 PDF: {INPUT.name}", flush=True)\nprint(f"📚 Páginas: {pages}", flush=True)\nprint(f"📦 Tamaño original: {fmt_bytes(INPUT.stat().st_size)}", flush=True)\nprint()\n\nprint("🔥 Inicializando modelos OCR...", flush=True)\nmodel_t0 = time.time()\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint(\n    f"✅ Modelos listos en {fmt_seconds(time.time() - model_t0)}.",\n    flush=True,\n)\nprint()\n\n# ---------------------------------------------------------------------\n# Process page by page.\n# ---------------------------------------------------------------------\nstarted = time.time()\npage_times = []\nocr_pages = 0\nskipped_pages = 0\ntotal_lines = 0\ninserted_lines = 0\nlow_score_lines = 0\n\nfor index in range(pages):\n    page_no = index + 1\n    page = doc[index]\n    page_t0 = time.time()\n\n    existing_text = page.get_text("text").strip()\n    existing_chars = len("".join(existing_text.split()))\n\n    if SKIP_SEARCHABLE and existing_chars >= EXISTING_TEXT_MIN:\n        skipped_pages += 1\n        duration = time.time() - page_t0\n        page_times.append(duration)\n\n        avg = statistics.mean(page_times[-8:])\n        eta = avg * (pages - page_no)\n\n        print(\n            f"[{ascii_bar(page_no, pages)}] "\n            f"{page_no:>4}/{pages} · {page_no/pages*100:5.1f}% · "\n            f"⏭ ya searchable ({existing_chars} chars) · "\n            f"ETA {fmt_seconds(eta)}",\n            flush=True,\n        )\n        continue\n\n    print(\n        f"▶ Página {page_no}/{pages} · render {DPI} DPI → OCR...",\n        flush=True,\n    )\n\n    # RGB, no alpha: recommended for OCR and lower memory use.\n    pix = page.get_pixmap(\n        dpi=DPI,\n        colorspace=pymupdf.csRGB,\n        alpha=False,\n        annots=True,\n    )\n\n    # Zero-copy-ish NumPy view over pixmap data, then make contiguous copy\n    # because Paddle may keep/use the array after this expression.\n    img = np.frombuffer(pix.samples, dtype=np.uint8)\n    img = img.reshape(pix.height, pix.width, pix.n)\n    if pix.n > 3:\n        img = img[:, :, :3]\n    img = np.ascontiguousarray(img)\n\n    ocr_t0 = time.time()\n    predictions = list(ocr.predict(img))\n    ocr_seconds = time.time() - ocr_t0\n\n    texts = []\n    scores = []\n    boxes = []\n\n    for prediction in predictions:\n        d = result_dict(prediction)\n        p_texts = list(d.get("rec_texts", []) or [])\n        p_scores = list(d.get("rec_scores", []) or [])\n        p_boxes = d.get("rec_boxes", [])\n\n        try:\n            p_boxes = list(p_boxes)\n        except Exception:\n            p_boxes = []\n\n        n = min(len(p_texts), len(p_scores), len(p_boxes))\n        texts.extend(p_texts[:n])\n        scores.extend(p_scores[:n])\n        boxes.extend(p_boxes[:n])\n\n    total_lines += len(texts)\n\n    fontname = install_page_font(page)\n    page_inserted = 0\n    page_low = 0\n\n    for text, score, box in zip(texts, scores, boxes):\n        try:\n            score = float(score)\n        except Exception:\n            score = 0.0\n\n        if score < MIN_SCORE:\n            page_low += 1\n            continue\n\n        text = clean_text(text)\n        if not text:\n            continue\n\n        rect = map_pix_box_to_page(page, pix, box)\n\n        if insert_hidden_text(page, rect, text, fontname):\n            page_inserted += 1\n\n    inserted_lines += page_inserted\n    low_score_lines += page_low\n    ocr_pages += 1\n\n    # Release big per-page buffers promptly.\n    del img\n    del pix\n    del predictions\n\n    duration = time.time() - page_t0\n    page_times.append(duration)\n\n    # Recent-page rolling average gives a more useful ETA than first-page warmup.\n    recent = page_times[-8:]\n    avg = statistics.mean(recent)\n    eta = avg * (pages - page_no)\n\n    print(\n        f"[{ascii_bar(page_no, pages)}] "\n        f"{page_no:>4}/{pages} · {page_no/pages*100:5.1f}% · "\n        f"OCR {ocr_seconds:4.1f}s · "\n        f"{page_inserted} líneas · "\n        f"página {duration:4.1f}s · "\n        f"ETA {fmt_seconds(eta)}",\n        flush=True,\n    )\n\n# ---------------------------------------------------------------------\n# Optimize font subsets if supported, then save as a NEW PDF.\n# ---------------------------------------------------------------------\nprint()\nprint("💾 Construyendo PDF con capa OCR invisible...", flush=True)\nsave_t0 = time.time()\n\ntry:\n    # Helps avoid carrying an entire Unicode font for every page.\n    doc.subset_fonts(verbose=False)\n    print("✅ Fuentes OCR subseteadas.", flush=True)\nexcept Exception as e:\n    print(f"ℹ️ subset_fonts omitido: {e}", flush=True)\n\nOUTPUT.parent.mkdir(parents=True, exist_ok=True)\n\ndoc.save(\n    OUTPUT,\n    garbage=4,\n    deflate=True,\n)\ndoc.close()\n\nsave_seconds = time.time() - save_t0\nelapsed = time.time() - started\n\nprint(\n    f"✅ PDF guardado en {fmt_seconds(save_seconds)}: {OUTPUT}",\n    flush=True,\n)\n\n# ---------------------------------------------------------------------\n# Final verification: reopen generated PDF and count extractable pages.\n# ---------------------------------------------------------------------\nprint()\nprint("🔎 Verificando capa searchable...", flush=True)\n\ncheck = pymupdf.open(OUTPUT)\nsearchable_pages = 0\nextractable_chars = 0\n\nfor p in check:\n    txt = p.get_text("text").strip()\n    chars = len("".join(txt.split()))\n    if chars > 0:\n        searchable_pages += 1\n        extractable_chars += chars\n\ncheck.close()\n\nif searchable_pages == 0:\n    raise RuntimeError(\n        "El PDF se generó, pero no encuentro texto extraíble. "\n        "No lo considero un resultado válido."\n    )\n\nprint()\nprint("=" * 78, flush=True)\nprint("✅ OCR PDF COMPLETO", flush=True)\nprint("=" * 78, flush=True)\nprint(f"Páginas totales:       {pages}", flush=True)\nprint(f"Páginas OCR procesadas:{ocr_pages:>8}", flush=True)\nprint(f"Páginas ya searchables:{skipped_pages:>8}", flush=True)\nprint(f"Líneas detectadas:     {total_lines:>8}", flush=True)\nprint(f"Líneas insertadas:     {inserted_lines:>8}", flush=True)\nprint(f"Líneas bajo score:     {low_score_lines:>8}", flush=True)\nprint(f"Páginas con texto final:{searchable_pages:>7}/{pages}", flush=True)\nprint(f"Caracteres extraíbles: {extractable_chars:>8}", flush=True)\nprint(f"Tiempo total:          {fmt_seconds(elapsed):>8}", flush=True)\nprint(f"Tamaño salida:         {fmt_bytes(OUTPUT.stat().st_size):>8}", flush=True)\nprint(f"Archivo: {OUTPUT}", flush=True)\nprint("✅ OUTPUT_READY", flush=True)', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "bos"
env["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)

cmd = [
    sys.executable,
    str(worker_path),
    "--input", str(input_path),
    "--output", str(output_path),
    "--dpi", str(DPI),
    "--min-score", str(MIN_SCORE),
    "--skip-searchable", "1" if SKIP_ALREADY_SEARCHABLE else "0",
    "--existing-text-min", str(EXISTING_TEXT_MIN_CHARS),
]

print("🧠 Iniciando OCR...")
print("💾 Cache de modelos:", MODEL_CACHE)
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

output_ready = False

for line in proc.stdout:
    print(line, end="", flush=True)
    if "✅ OUTPUT_READY" in line:
        output_ready = True

rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"El OCR falló con código {rc}.")

if not output_ready or not output_path.exists():
    raise RuntimeError(
        "El worker terminó pero no confirmó un PDF de salida válido."
    )

print()
print("⬇️ Descargando automáticamente:")
print(output_path.name)

try:
    files.download(str(output_path))
except Exception as e:
    print()
    print("⚠️ La descarga automática no pudo iniciarse:", e)
    print("Te dejo un enlace local como fallback:")
    from IPython.display import display, FileLink
    display(FileLink(str(output_path)))